In [1]:
# A default setup cell.
# It imports environment variables, define 'devtools.debug" as a buildins, set PYTHONPATH, and code auto-reload
# Copy it in other Notebooks

%load_ext autoreload
%autoreload 2
%reset -f

from devtools import debug  # noqa: F401  # noqa: F811
from dotenv import load_dotenv
from rich import print  # noqa: F401

assert load_dotenv(verbose=True)

In [2]:
from src.utils.config_mngr import global_config, global_config_reload

global_config_reload()

list_demos = global_config().merge_with("config/demos/document_extractor.yaml").get_list("Document_extractor_demo")


test_schema = next((item for item in list_demos if item.get("schema_name") == "Rainbow File"))


In [3]:
from src.demos.ekg.pydantic_rag import PydanticRag

KV_STORE = "file"


vector_store = PydanticRag.get_vector_store()

rag = PydanticRag(
    model_definition=test_schema,
    vector_store=vector_store,
    llm_id=None,
    kv_store_id=KV_STORE,
)


2025-08-14 01:01:46.984 | INFO     | src.ai_extra.pgvector_factory:create_pg_vector_store:77 - Hybrid search enabled with config: HybridSearchConfig(tsv_column='content_tsv', tsv_lang='pg_catalog.english', fts_query='', fusion_function=<function reciprocal_rank_fusion at 0x7057fad3cf40>, fusion_function_parameters={}, primary_top_k=4, secondary_top_k=4, index_name='pydantic_fields_qwen3_06b_tsv_index', index_type='GIN')
2025-08-14 01:01:47.035 | DEBUG    | src.ai_extra.pgvector_factory:create_pg_vector_store:93 - Use existing pgvector table : pydantic_fields_qwen3_06b


src/ai_extra/pgvector_factory.py:107 create_pg_vector_store
    vector_store: <langchain_postgres.v2.vectorstores.PGVectorStore object at 0x7057f99f4590> (PGVectorStore)


2025-08-14 01:01:47.085 | WARNING  | src.ai_extra.pgvector_factory:create_pg_vector_store:116 - Failed to apply hybrid search index: 'PGVectorStore' object has no attribute 'apply_hybrid_search_index'
2025-08-14 01:01:47.086 | INFO     | src.ai_core.vector_store_factory:get:230 - get vector store  : PgVector/pydantic_fields_qwen3_06b cache_embeddings: False
2025-08-14 01:01:47.136 | DEBUG    | src.ai_core.llm_factory:get_llm:530 - get LLM:'kimi_k2_openrouter' -extra: {'temperature': 0.0}


In [4]:
import os
from pathlib import Path

doc_id = "03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2_extracted.json"

file1 = Path(os.getenv("ONEDRIVE", "")) / "prj/atos-kg/rainbow-json/" / doc_id
assert file1.exists()
doc_text = file1.read_text()

rainbow_report = rag.analyze_document(
    document_id=doc_id,
    markdown=doc_text,
)

print("Structured result:", rainbow_report)

assert rainbow_report


2025-08-14 01:01:47.667 | DEBUG    | src.utils.pydantic.kv_store:load_object_from_kvstore:89 - read 'RainbowProjectAnalysis/03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2_extracted.json.json' from KV store


Structured result:
RainbowProjectAnalysis(
    identification=ProjectIdentification(
        name='CNES TMA VENUS VIP_PEPS_ THEIA MUSCATE',
        opportunity_id='9000559500',
        customer='CNES',
        customer_segment='Aerospace',
        status='Solution Review',
        start_date='2019-04-01',
        end_date='2022-03-31'
    ),
    description=ProjectDescription(
        objectives=[
            'Tierce Maintenance Applicative (TMA) of CNES Earth-observation data production centres and their 
image-processing chains',
            'Corrective and evolutionary maintenance of three lots: VENUS VIP, PEPS, THEIA MUSCATE'
        ],
        scope='Take-over phase followed by corrective/evolutionary maintenance for three lots (mono-award 
framework)',
        success_metrics=['SLA compliance', 'Quality of deliverables', 'Customer satisfaction'],
        differentiators=[
            'Existing incumbent on PEPS',
            'Deep Earth-observation domain knowledge',
            'MUNDI platform synergies'
        ]
    ),
    team=[
        PersonRole(name='Marc Ferrer', role='Global Client Leader', organization='Atos', contact_type='internal'),
        PersonRole(
            name='Aurore Dorez',
            role='Sales Lead / Deal Maker',
            organization='Atos',
            contact_type='internal'
        ),
        PersonRole(name='Barthélémy Marti', role='Bid Manager', organization='Atos', contact_type='internal'),
        PersonRole(
            name='Olivier Rondeau',
            role='Solution Manager / Contract Executive',
            organization='Atos',
            contact_type='internal'
        ),
        PersonRole(name='Caroline Jaulin', role='Finance Lead', organization='Atos', contact_type='internal'),
        PersonRole(name='Danièle Phankongsy', role='Deal Lawyer', organization='Atos', contact_type='internal'),
        PersonRole(name='Sonia Gouel', role='Project Manager', organization='Atos', contact_type='internal'),
        PersonRole(
            name='Pierre Bourrousse',
            role='Strategy Achat Sponsor',
            organization='CNES',
            contact_type='external'
        ),
        PersonRole(name='Gérard Lassalle-Valler', role='Sponsor', organization='CNES', contact_type='external'),
        PersonRole(
            name='Sylvia Sylvander',
            role='CP CNES Décideur',
            organization='CNES',
            contact_type='external'
        )
    ],
    delivery=DeliveryInfo(
        business_lines=['B1P S BS Toulouse'],
        locations=['Toulouse, France'],
        partners=[
            'ACS (subcontractor for Venus VIP maintenance, imposed by CNES)',
            'PIXSTART (subcontractor for THEIA MUSCATE first-year fixes and knowledge transfer)'
        ],
        technologies=[
            'VENUS VIP image-processing chains',
            'PEPS Sentinel data exploitation platform',
            'THEIA MUSCATE continental-surface data services'
        ]
    ),
    financials=FinancialMetrics(
        tcv=1800000.0,
        annual_revenue=300000.0,
        project_margin=21.0,
        payment_terms='30 days from invoice date, quarterly invoicing'
    ),
    risks=[
        RiskAnalysis(
            risk_description='Penalties due to SLA non-compliance caused by quality or delivery delays',
            impact_level='high',
            mitigation_strategy='Rigorous QA and project management',
            status='open'
        ),
        RiskAnalysis(
            risk_description='Cost overruns from underestimation of rework, missing packages, or non-representative
platforms',
            impact_level='medium',
            mitigation_strategy='Detailed due-diligence and contingency planning',
            status='open'
        ),
        RiskAnalysis(
            risk_description='Competency retention issues due to turnover and declining activity',
            impact_level='medium',
            mitigation_strategy='Knowledge management and retention incentives'

In [5]:
from src.utils.pydantic.kv_store import save_object_to_kvstore

save_object_to_kvstore(doc_id, rainbow_report)

2025-08-14 01:01:47.740 | DEBUG    | src.utils.pydantic.kv_store:save_object_to_kvstore:63 - add key 'RainbowProjectAnalysis/03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2_extracted.json.json' to kv_store <langchain.storage.file_system.LocalFileStore object at 0x7057d8194320>'


In [6]:
rag.get_key_description()

'Unique identifier from Salesforce or similar CRM'

In [7]:
debug(rainbow_report)

/tmp/ipykernel_39760/1838901977.py:1 <module>
    rainbow_report: RainbowProjectAnalysis(
        identification=ProjectIdentification(
            name='CNES TMA VENUS VIP_PEPS_ THEIA MUSCATE',
            opportunity_id='9000559500',
            customer='CNES',
            customer_segment='Aerospace',
            status='Solution Review',
            start_date='2019-04-01',
            end_date='2022-03-31',
        ),
        description=ProjectDescription(
            objectives=[
                (
                    'Tierce Maintenance Applicative (TMA) of CNES Earth-observation data production centres and their '
                    'image-processing chains'
                ),
                'Corrective and evolutionary maintenance of three lots: VENUS VIP, PEPS, THEIA MUSCATE',
            ],
            scope='Take-over phase followed by corrective/evolutionary maintenance for three lots (mono-award framework)',
            success_metrics=[
                'SLA compliance

RainbowProjectAnalysis(identification=ProjectIdentification(name='CNES TMA VENUS VIP_PEPS_ THEIA MUSCATE', opportunity_id='9000559500', customer='CNES', customer_segment='Aerospace', status='Solution Review', start_date='2019-04-01', end_date='2022-03-31'), description=ProjectDescription(objectives=['Tierce Maintenance Applicative (TMA) of CNES Earth-observation data production centres and their image-processing chains', 'Corrective and evolutionary maintenance of three lots: VENUS VIP, PEPS, THEIA MUSCATE'], scope='Take-over phase followed by corrective/evolutionary maintenance for three lots (mono-award framework)', success_metrics=['SLA compliance', 'Quality of deliverables', 'Customer satisfaction'], differentiators=['Existing incumbent on PEPS', 'Deep Earth-observation domain knowledge', 'MUNDI platform synergies']), team=[PersonRole(name='Marc Ferrer', role='Global Client Leader', organization='Atos', contact_type='internal'), PersonRole(name='Aurore Dorez', role='Sales Lead / De

In [8]:
chunks = rag.chunck(rainbow_report)
debug(chunks)


/tmp/ipykernel_39760/280282971.py:2 <module>
    chunks: [
        Document(
            metadata={
                'field_name': 'identification',
                'model_class': 'RainbowProjectAnalysis',
                'description': 'Project identification information',
                'document_id': '03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2_extracted.json',
            },
            page_content=(
                '# name\n'
                'CNES TMA VENUS VIP_PEPS_ THEIA MUSCATE\n'
                '# opportunity_id\n'
                '9000559500\n'
                '# customer\n'
                'CNES\n'
                '# customer_segment\n'
                'Aerospace\n'
                '# status\n'
                'Solution Review\n'
                '# start_date\n'
                '2019-04-01\n'
                '# end_date\n'
                '2022-03-31'
            ),
        ),
        Document(
            metadata={
                'field_name': 'des

[Document(metadata={'field_name': 'identification', 'model_class': 'RainbowProjectAnalysis', 'description': 'Project identification information', 'document_id': '03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2_extracted.json'}, page_content='# name\nCNES TMA VENUS VIP_PEPS_ THEIA MUSCATE\n# opportunity_id\n9000559500\n# customer\nCNES\n# customer_segment\nAerospace\n# status\nSolution Review\n# start_date\n2019-04-01\n# end_date\n2022-03-31'),
 Document(metadata={'field_name': 'description', 'model_class': 'RainbowProjectAnalysis', 'description': 'Project characteristics description', 'document_id': '03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2_extracted.json'}, page_content='- objectives\n  - Tierce Maintenance Applicative (TMA) of CNES Earth-observation data production centres and their image-processing chains\n  - Corrective and evolutionary maintenance of three lots: VENUS VIP, PEPS, THEIA MUSCATE\n# scope\nTake-over phase followed by correcti

In [9]:
"\n -". join([f"{k}: {v}"  for k,v in rag.get_top_class_fields().items()])

'identification: Project identification information\n -description: Project characteristics description\n -team: All involved personnel\n -delivery: Operational delivery information\n -financials: Financial metrics\n -risks: Risk analysis\n -competition: Competitive landscape\n -bidding: Bidding strategy\n -similarity: Similarity search attributes\n -source: Source document metadata'

In [10]:
from langchain_core.utils.function_calling import convert_to_openai_tool

dyn_tool = rag.create_vector_search_tool()
debug(convert_to_openai_tool(dyn_tool))


/tmp/ipykernel_39760/1577126679.py:4 <module>
    convert_to_openai_tool(dyn_tool): {
        'type': 'function',
        'function': {
            'name': 'RainbowProjectAnalysis_retriever',
            'description': (
                '\n'
                "Retrieve information related to documents described as 'Input document for a review meeting (called '"
                "Rainbow') that assess an opportunity for a project and the business proposal, before it's sent to the"
                ' customer..\n'
                'The mandatory argument are:\n'
                '- query.  \n'
                '- selected_section: a list of section names that best match the query. Select several if you are not '
                'sure.\n'
                'Additional arguments: \n'
                '- entity_key:  an id to limit search to a given doc \n'
            ),
            'parameters': {
                'properties': {
                    'query': {
                        'description': 

{'type': 'function',
 'function': {'name': 'RainbowProjectAnalysis_retriever',
  'description': "\nRetrieve information related to documents described as 'Input document for a review meeting (called 'Rainbow') that assess an opportunity for a project and the business proposal, before it's sent to the customer..\nThe mandatory argument are:\n- query.  \n- selected_section: a list of section names that best match the query. Select several if you are not sure.\nAdditional arguments: \n- entity_key:  an id to limit search to a given doc \n",
  'parameters': {'properties': {'query': {'description': 'The query',
     'type': 'string'},
    'selected_section': {'description': "\nList of sections relevant for the query.\n\nAllowed section name SHOULD BE in that list: \n- 'identification' (Project identification information)\n- 'description' (Project characteristics description)\n- 'team' (All involved personnel)\n- 'delivery' (Operational delivery information)\n- 'financials' (Financial metric

In [11]:
dyn_tool.invoke({"query": "hello", "selected_section": ["bidding"]})

'# strategy_type\nFertilisation\n- win_themes\n  - Valorisation of Earth-observation expertise\n  - Commercial valorisation of CNES tools\n  - Synergies with MUNDI and H2020\n# pricing_strategy\nOffer free transitions for THEIA MUSCATE and Venus VIP\n- challenges\n  - ACS imposed subcontractor on Venus VIP\n  - Strong incumbent CAP on THEIA MUSCATE'

In [18]:
r = dyn_tool.invoke({"query": "hello", "selected_section": ["bidding"]})
print(r)

# strategy_type
Fertilisation
- win_themes
  - Valorisation of Earth-observation expertise
  - Commercial valorisation of CNES tools
  - Synergies with MUNDI and H2020
# pricing_strategy
Offer free transitions for THEIA MUSCATE and Venus VIP
- challenges
  - ACS imposed subcontractor on Venus VIP
  - Strong incumbent CAP on THEIA MUSCATE
# strategy_type
Fertilisation
- win_themes
  - Valorisation of Earth-observation expertise
  - Commercial valorisation of CNES tools
  - Synergies with MUNDI and H2020
# pricing_strategy
Offer free transitions for THEIA MUSCATE and Venus VIP
- challenges
  - ACS imposed subcontractor on Venus VIP
  - Strong incumbent CAP on THEIA MUSCATE

In [12]:
# 2. Index the document
rag.store_chunks(chunks)
print("Document stored.")


Document stored.

In [13]:
hits = rag.query_vectorstore("e-mail address", k=2)
print("Vector hits:", hits)

Vector hits:
[
    Document(
        id='79320347-6524-4ee2-b0c8-00dbb5a35da3',
        metadata={
            'description': 'Project identification information',
            'document_id': '03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2_extracted.json',
            'field_name': 'identification',
            'model_class': 'RainbowProjectAnalysis'
        },
        page_content='# name\nCNES TMA VENUS VIP_PEPS_ THEIA MUSCATE\n# opportunity_id\n9000559500\n# 
customer\nCNES\n# customer_segment\nAerospace\n# status\nSolution Review\n# start_date\n2019-04-01\n# 
end_date\n2022-03-31'
    ),
    Document(
        id='ba2300e0-56d7-459b-954f-b51ee5656e65',
        metadata={
            'description': 'Project identification information',
            'document_id': '03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2_extracted.json',
            'field_name': 'identification',
            'model_class': 'RainbowProjectAnalysis'
        },
        page_content='# name\nCNES TMA VENUS VIP_PEPS_ THEIA MUSCATE\n# opportunity_id\n9000559500\n# 
customer\nCNES\n# customer_segment\nAerospace\n# status\nSolution Review\n# start_date\n2019-04-01\n# 
end_date\n2022-03-31'
    )
]

In [14]:
# 3. Query the vector store
hits = rag.query_vectorstore("revenue", k=2, filter={"field_name": {"$eq": "financials"}})
print("Vector hits:", hits)

Vector hits:
[
    Document(
        id='8813e449-5cbe-425c-b1ea-a7fefedadf26',
        metadata={
            'description': 'Financial metrics',
            'document_id': '03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2_extracted.json',
            'field_name': 'financials',
            'model_class': 'RainbowProjectAnalysis'
        },
        page_content='# tcv\n1800000.0\n# annual_revenue\n300000.0\n# project_margin\n21.0\n# payment_terms\n30 
days from invoice date, quarterly invoicing'
    ),
    Document(
        id='3a83b3c3-b316-42a2-bbc9-e57c8f5b7d8a',
        metadata={
            'description': 'Financial metrics',
            'document_id': '03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2_extracted.json',
            'field_name': 'financials',
            'model_class': 'RainbowProjectAnalysis'
        },
        page_content='# tcv\n1800000.0\n# annual_revenue\n300000.0\n# project_margin\n21.0\n# payment_terms\n30 
days from invoice date, quarterly invoicing'
    )
]